# Matrix-Factorization Recommender System

This notebook derives a regularized low-rank recommender objective, implements alternating least squares (ALS) and stochastic gradient descent (SGD), and validates a small observed-entry problem with SciPy L-BFGS-B. Ratings come from the real [MovieLens latest-small dataset](https://grouplens.org/datasets/movielens/); the download cell records the source and keeps the experiment reproducible by fixing a seed.

## 1. Objective and alternating minimization

For observed ratings $\mathcal O$, user factors $U\in\mathbb R^{m\times k}$ and item factors $V\in\mathbb R^{n\times k}$ solve

$$\min_{U,V}\; F(U,V)=\frac12\sum_{(u,i)\in\mathcal O}(r_{ui}-U_u^TV_i)^2+\frac{\lambda}{2}(\|U\|_F^2+\|V\|_F^2).$$

The problem is non-convex jointly but convex in either block. Holding $V$ fixed gives the normal equations

$$(V_{\mathcal I_u}^TV_{\mathcal I_u}+\lambda I)U_u=V_{\mathcal I_u}^Tr_u,$$

and an analogous equation for each item. These exact block minimizers explain ALS. SGD uses the per-rating gradients $\partial F/\partial U_u=(U_u^TV_i-r_{ui})V_i+\lambda U_u$ and the symmetric item expression. Neither method has a global guarantee for the joint non-convex objective; ALS is monotone for exact block updates.

## 2. Real data and experiment design

The official GroupLens archive is downloaded once by this cell. We select active users/items, hold out one rating per user for a test set, and report RMSE on observed test ratings. The regularized objective—not only RMSE—is compared across ALS, SGD and an independent L-BFGS-B reference on a compact subset.

In [ ]:
import io, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
ratings = pd.read_csv(archive.open('ml-latest-small/ratings.csv'))
rng = np.random.default_rng(2026)
ratings = ratings[ratings.userId.isin(ratings.userId.value_counts().head(80).index)]
ratings = ratings[ratings.movieId.isin(ratings.movieId.value_counts().head(120).index)].copy()
user_ids = {value:index for index,value in enumerate(sorted(ratings.userId.unique()))}
item_ids = {value:index for index,value in enumerate(sorted(ratings.movieId.unique()))}
ratings['u'] = ratings.userId.map(user_ids); ratings['i'] = ratings.movieId.map(item_ids)
indices = np.arange(len(ratings)); rng.shuffle(indices); test_idx = indices[:len(indices)//5]; train_idx = indices[len(indices)//5:]
train = ratings.iloc[train_idx][['u','i','rating']].to_numpy(float); test = ratings.iloc[test_idx][['u','i','rating']].to_numpy(float)
m, n, k, lam = len(user_ids), len(item_ids), 12, 0.10
m, n, len(train), len(test)

In [ ]:
def observed_loss(U, V, sample):
    errors = sample[:, 2] - np.sum(U[sample[:, 0].astype(int)] * V[sample[:, 1].astype(int)], axis=1)
    return 0.5 * errors @ errors + 0.5 * lam * (U @ U).sum() + 0.5 * lam * (V @ V).sum()

def als(train, m, n, k, epochs=25):
    U = rng.normal(0, 0.1, (m, k)); V = rng.normal(0, 0.1, (n, k)); history=[]
    for _ in range(epochs):
        for u in range(m):
            rows = train[train[:,0] == u]; A = V[rows[:,1].astype(int)]
            U[u] = np.linalg.solve(A.T @ A + lam*np.eye(k), A.T @ rows[:,2]) if len(rows) else U[u]
        for i in range(n):
            rows = train[train[:,1] == i]; A = U[rows[:,0].astype(int)]
            V[i] = np.linalg.solve(A.T @ A + lam*np.eye(k), A.T @ rows[:,2]) if len(rows) else V[i]
        history.append(observed_loss(U,V,train))
    return U,V,np.asarray(history)

U_als, V_als, als_history = als(train, m, n, k)
plt.plot(als_history, marker='o'); plt.xlabel('ALS epoch'); plt.ylabel('Regularized training objective'); plt.title('Block-coordinate descent'); plt.tight_layout();

In [ ]:
def sgd(train, m, n, k, epochs=30, learning_rate=0.02):
    U = rng.normal(0, 0.1, (m, k)); V = rng.normal(0, 0.1, (n, k)); history=[]
    for epoch in range(epochs):
        order = rng.permutation(len(train))
        for row in train[order]:
            u, i, rating = int(row[0]), int(row[1]), row[2]
            error = U[u] @ V[i] - rating
            U[u] -= learning_rate * (error * V[i] + lam * U[u])
            V[i] -= learning_rate * (error * U[u] + lam * V[i])
        history.append(observed_loss(U,V,train))
    return U,V,np.asarray(history)

U_sgd, V_sgd, sgd_history = sgd(train, m, n, k)
plt.plot(sgd_history, label='SGD'); plt.plot(als_history, label='ALS'); plt.xlabel('Epoch'); plt.ylabel('Objective'); plt.legend(); plt.tight_layout();

In [ ]:
def rmse(U,V,sample):
    pred = np.sum(U[sample[:,0].astype(int)] * V[sample[:,1].astype(int)], axis=1)
    return np.sqrt(np.mean((sample[:,2]-pred)**2))

small = train[:min(600, len(train))]; q = min(6, k); small_m, small_n = m, n
def reference_objective(x):
    U=x[:small_m*q].reshape(small_m,q); V=x[small_m*q:].reshape(small_n,q)
    return observed_loss(U,V,small)
x0=np.r_[U_als[:,:q].ravel(),V_als[:,:q].ravel()]
reference=minimize(reference_objective,x0,method='L-BFGS-B',options={'maxiter':150})
print({'ALS test RMSE':rmse(U_als,V_als,test),'SGD test RMSE':rmse(U_sgd,V_sgd,test),'L-BFGS-B objective':reference.fun,'L-BFGS-B success':reference.success})